In [2]:
import nibabel as nib
import os
import matplotlib.pyplot as plt
import numpy as np
from dipy.io.gradients import read_bvals_bvecs
from fury import window, actor

# Data and Class imports
# from data import data, affine, labels, bvals, bvecs, vox_size
# from env_main_updated import UnifiedTractographyEnv
# from agent import BranchingStreamlineAgent,UnifiedBranchingAgent
# from dipy.viz import colormap
# from dipy.tracking import utils

In [3]:
mask_path="sub-1001\mask"
data=[]
for i,mask in enumerate(os.listdir(mask_path)):
    mask_file=os.path.join(mask_path,mask)
    print(mask_file)
    data.append(nib.load(mask_file).get_fdata())
    data[i]=data[i].astype(bool)

sub-1001\mask\sub-1001__mask_csf.nii.gz
sub-1001\mask\sub-1001__mask_gm.nii.gz
sub-1001\mask\sub-1001__mask_wm.nii.gz


In [4]:
data[0].shape

(135, 166, 100)

In [5]:
coords=np.argwhere((data[2]==True) & (data[2]==True))
print(coords.shape)

(382567, 3)


In [33]:
mid=data[2].shape[2]//2
seeds=data[2][:,:,:mid-35]
seed_coords=np.argwhere(seeds)
print(seed_coords.shape)

(46458, 3)


In [27]:
scene = window.Scene()
points=np.array(seed_coords)
point_actor=actor.point(points,colors=(1,0,0),point_radius=0.25)
scene.add(point_actor)
window.show(scene,size=(800,800))

In [35]:
n_samples=10000
np.random.shuffle(seed_coords)
indices = np.random.choice(seed_coords.shape[0], n_samples, replace=False)
reduced_matrix = coords[indices]
reduced_matrix.shape

(10000, 3)

In [36]:
scene = window.Scene()
points=np.array(reduced_matrix)
point_actor=actor.point(points,colors=(1,0,0),point_radius=0.5)
scene.add(point_actor)
window.show(scene,size=(800,800))

In [5]:
peaks_data=nib.load(r"sub-1001\fodf\sub-1001__peaks.nii.gz").get_fdata()
peaks_data.shape

(135, 166, 100, 15)

In [10]:
import os
import subprocess
import sys

# 1. Use absolute paths to be 100% sure
base_dir = os.path.abspath("sub-1001")
peaks_path = os.path.join(base_dir, "fodf", "sub-1001__peaks.nii.gz")
output_dir = os.path.join(base_dir, "subdivisions")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def run_segmentation(input_peaks, out_folder):
    # Using 'python -m tractseg.python_api' is the most stable way on Windows
    cmd_base = [sys.executable, "-m", "tractseg.python_api", "-i", input_peaks, "-o", out_folder]
    
    print(f"Checking input file: {os.path.exists(input_peaks)}")
    
    # Run Major Bundles
    print("Running Major Bundle Segmentation...")
    subprocess.run(cmd_base + ["--output_type", "tract_segmentation", "--verbose"], shell=True)

    # Run CC Subdivisions
    print("Running CC Segmentation...")
    subprocess.run(cmd_base + ["--output_type", "central_occlusion_segmentation", "--verbose"], shell=True)

    # List files to verify
    print("\nVerifying output files...")
    for root, dirs, files in os.walk(out_folder):
        for file in files:
            print(f"Found: {os.path.join(root, file)}")

if __name__ == "__main__":
    run_segmentation(peaks_path, output_dir)

Checking input file: True
Running Major Bundle Segmentation...
Running CC Segmentation...

Verifying output files...


In [11]:
import os
import subprocess
import sys

# Absolute paths are safer on Windows
base_dir = os.path.abspath("sub-1001")
peaks_path = os.path.join(base_dir, "fodf", "sub-1001__peaks.nii.gz")
output_dir = os.path.join(base_dir, "subdivisions")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def run_tractseg_fixed(input_peaks, out_folder):
    # --preprocess: Aligns your native peaks to MNI space so the CNN works
    # --verbose: Shows us exactly what the network is thinking
    cmd = [
        sys.executable, "-m", "tractseg.python_api", 
        "-i", input_peaks, 
        "-o", out_folder, 
        "--output_type", "tract_segmentation",
        "--preprocess", 
        "--verbose"
    ]
    
    print(f"Starting TractSeg on: {input_peaks}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    # Print the output so we can see why it might be failing
    print("STDOUT:", result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)

    # Check if files were created
    expected_path = os.path.join(out_folder, "bundle_segmentations")
    if os.path.exists(expected_path) and os.listdir(expected_path):
        print(f"Success! Files found in: {expected_path}")
    else:
        print("Folder is still empty. Possible peak scaling issue.")

run_tractseg_fixed(peaks_path, output_dir)


Starting TractSeg on: c:\Users\Samsung\Desktop\tractography\main\sub-1001\fodf\sub-1001__peaks.nii.gz
STDOUT: 
Folder is still empty. Possible peak scaling issue.


In [12]:
import nibabel as nib
import os

# Paths
peaks_path = r"sub-1001\fodf\sub-1001__peaks.nii.gz"
wm_mask_path = r"sub-1001\mask\sub-1001__mask_wm.nii.gz"
fixed_peaks_path = r"sub-1001\fodf\sub-1001__peaks_fixed.nii.gz"

# 1. Load the peaks and the mask (which has the correct affine)
peaks_img = nib.load(peaks_path)
wm_img = nib.load(wm_mask_path)

# 2. Check if affines match
if not (peaks_img.affine == wm_img.affine).all():
    print("Affine mismatch detected! Fixing...")
    # Create a new image using peak data but the WM mask's affine/header
    fixed_peaks = nib.Nifti1Image(peaks_img.get_fdata(), wm_img.affine, wm_img.header)
    nib.save(fixed_peaks, fixed_peaks_path)
    print(f"Fixed peaks saved to: {fixed_peaks_path}")
else:
    print("Affines already match. The issue might be Peak Scaling.")

Affines already match. The issue might be Peak Scaling.


In [14]:
# Normalize peaks so the maximum value is 1.0
fixed_peaks = nib.Nifti1Image(peaks_img.get_fdata(), wm_img.affine, wm_img.header)
data = fixed_peaks.get_fdata()
data = data / (np.max(data) + 1e-8) 
normalized_peaks = nib.Nifti1Image(data, wm_img.affine, wm_img.header)
nib.save(normalized_peaks, "normalized_peaks.nii.gz")

In [15]:
import numpy as np
import nibabel as nib

# Load your WM mask
wm_img = nib.load(r"sub-1001\mask\sub-1001__mask_wm.nii.gz")
wm_data = wm_img.get_fdata()

# The CC is always in the center of the X-axis (sagittal plane)
# In a 135-voxel width, the center is ~67
cc_seed_mask = np.zeros_like(wm_data)
# Slice a 4-voxel thick window in the middle
cc_seed_mask[65:69, :, :] = wm_data[65:69, :, :] 

# Save this as your "Manual CC Subdivision"
seed_img = nib.Nifti1Image(cc_seed_mask.astype(np.uint8), wm_img.affine)
nib.save(seed_img, "manual_cc_seeds.nii.gz")

In [16]:
seeds_data = seed_img.get_fdata()
seeds_data.shape

(135, 166, 100)

In [17]:
seeds_data.astype(bool)
coords=np.argwhere(seeds_data)

In [18]:
scene = window.Scene()
points=np.array(coords)
point_actor=actor.point(points,colors=(1,0,0),point_radius=0.5)
scene.add(point_actor)
window.show(scene,size=(800,800))

In [19]:
import nibabel as nib
import numpy as np
import os
import subprocess
import sys

# Paths
peaks_path = r"sub-1001\fodf\sub-1001__peaks.nii.gz"
wm_mask_path = r"sub-1001\mask\sub-1001__mask_wm.nii.gz"
output_dir = r"sub-1001\subdivisions"
temp_peaks = r"sub-1001\fodf\peaks_normalized.nii.gz"

def fix_and_run():
    # --- STEP 1: LOAD AND NORMALIZE ---
    print("Loading and normalizing peaks...")
    p_img = nib.load(peaks_path)
    m_img = nib.load(wm_mask_path)
    
    p_data = p_img.get_fdata()
    
    # Normalize peak lengths to [0, 1]
    # This is often why TractSeg produces empty folders
    p_max = np.max(np.abs(p_data))
    if p_max > 0:
        p_data = p_data / p_max

    # Use the WM mask's affine to ensure perfect spatial alignment
    fixed_img = nib.Nifti1Image(p_data.astype(np.float32), m_img.affine, m_img.header)
    nib.save(fixed_img, temp_peaks)

    # --- STEP 2: RUN TRACTSEG WITH PREPROCESS ---
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print("Running TractSeg on normalized data...")
    # --preprocess is critical here to handle the internal MNI transform
    cmd = [
        sys.executable, "-m", "tractseg.python_api",
        "-i", temp_peaks,
        "-o", output_dir,
        "--output_type", "tract_segmentation",
        "--preprocess",
        "--verbose"
    ]
    
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    print(result.stdout)
    if result.stderr:
        print("Errors/Warnings:", result.stderr)

    # Verify output
    bundle_path = os.path.join(output_dir, "bundle_segmentations")
    if os.path.exists(bundle_path) and os.listdir(bundle_path):
        print(f"Success! Anatomical masks generated in {bundle_path}")
    else:
        print("Still empty. It's possible the peaks need to be re-ordered to (X, Y, Z, 9).")

if __name__ == "__main__":
    fix_and_run()

Loading and normalizing peaks...
Running TractSeg on normalized data...

Still empty. It's possible the peaks need to be re-ordered to (X, Y, Z, 9).


In [20]:
import nibabel as nib
import numpy as np
import os
import subprocess
import sys

# Paths
peaks_path = r"sub-1001\fodf\sub-1001__peaks.nii.gz"
wm_mask_path = r"sub-1001\mask\sub-1001__mask_wm.nii.gz"
output_dir = r"sub-1001\subdivisions"
temp_peaks = r"sub-1001\fodf\peaks_9channel_normalized.nii.gz"

def fix_channels_and_run():
    print("Loading data...")
    p_img = nib.load(peaks_path)
    m_img = nib.load(wm_mask_path)
    
    p_data = p_img.get_fdata() # Shape: (135, 166, 100, 15)
    
    # --- STEP 1: CHANNEL SLICING ---
    # TractSeg expects exactly 9 channels (3 peaks x 3 coords)
    print(f"Original shape: {p_data.shape}. Slicing to 9 channels...")
    p_data_9 = p_data[..., :9] 
    
    # --- STEP 2: NORMALIZATION ---
    # Scaling to [0, 1] range based on max peak intensity
    p_max = np.max(np.abs(p_data_9))
    if p_max > 0:
        p_data_9 = p_data_9 / p_max

    # Save fixed version with WM mask's affine
    fixed_img = nib.Nifti1Image(p_data_9.astype(np.float32), m_img.affine, m_img.header)
    nib.save(fixed_img, temp_peaks)

    # --- STEP 3: RUN SEGMENTATION ---
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print("Running TractSeg on 9-channel data...")
    cmd = [
        sys.executable, "-m", "tractseg.python_api",
        "-i", temp_peaks,
        "-o", output_dir,
        "--output_type", "tract_segmentation",
        "--preprocess" # Critical for handling MNI space internally
    ]
    
    # Running with shell=True for Windows compatibility
    subprocess.run(cmd, shell=True)

    # Check for bundle_segmentations folder
    result_path = os.path.join(output_dir, "bundle_segmentations")
    if os.path.exists(result_path) and len(os.listdir(result_path)) > 0:
        print(f"SUCCESS! Masks generated in: {result_path}")
    else:
        print("Folder still empty. Ensure you have installed 'fslpy' and 'mrtrix3' if possible.")

if __name__ == "__main__":
    fix_channels_and_run()

Loading data...
Original shape: (135, 166, 100, 15). Slicing to 9 channels...
Running TractSeg on 9-channel data...
Folder still empty. Ensure you have installed 'fslpy' and 'mrtrix3' if possible.


In [21]:
import nibabel as nib
import numpy as np
import os
import subprocess
import sys

# Absolute Paths
base_dir = os.path.abspath("sub-1001")
peaks_path = os.path.join(base_dir, "fodf", "sub-1001__peaks.nii.gz")
wm_mask_path = os.path.join(base_dir, "mask", "sub-1001__mask_wm.nii.gz")
output_dir = os.path.join(base_dir, "subdivisions")
temp_peaks = os.path.join(base_dir, "fodf", "peaks_final_fix.nii.gz")

def final_attempt_segmentation():
    print("Loading peaks...")
    p_img = nib.load(peaks_path)
    m_img = nib.load(wm_mask_path)
    
    # Slice to 9 channels and ensure float32
    p_data = p_img.get_fdata()[..., :9].astype(np.float32)
    
    # --- CRITICAL: PEAK NORMALIZATION ---
    # TractSeg CNNs are sensitive to intensity. 
    # We scale so the 95th percentile of the 1st peak is 1.0.
    first_peak_lengths = np.linalg.norm(p_data[..., :3], axis=-1)
    scale_factor = np.percentile(first_peak_lengths[first_peak_lengths > 0], 95)
    p_data /= (scale_factor + 1e-8)
    
    print(f"Data scaled by {scale_factor:.4f}. Saving temporary peaks...")
    
    # Save with the WM mask's affine to ensure world-space alignment
    fixed_img = nib.Nifti1Image(p_data, m_img.affine, m_img.header)
    nib.save(fixed_img, temp_peaks)

    # --- RUN TRACTSEG ---
    # We add --n_cpus to ensure it's not a threading hang
    # We add --keep_intermediate to see what it's doing
    cmd = [
        sys.executable, "-m", "tractseg.python_api",
        "-i", temp_peaks,
        "-o", output_dir,
        "--output_type", "tract_segmentation",
        "--preprocess", 
        "--verbose"
    ]
    
    print("Executing TractSeg...")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    print("LOG OUTPUT:\n", result.stdout)
    
    if "Segmentation done" in result.stdout:
        print("Model finished. Checking for files...")
    else:
        print("Model failed to find any bundles. Errors:\n", result.stderr)

if __name__ == "__main__":
    final_attempt_segmentation()

Loading peaks...
Data scaled by 1.0000. Saving temporary peaks...
Executing TractSeg...
LOG OUTPUT:
 
Model failed to find any bundles. Errors:
 
